# Current deployed vs Ensemble regime

Head-to-head backtest on 41 days of 15m HL data across **31 symbols** (8 live + 23 expansion).

- **A (current)** — each live symbol runs its deployed config exactly as-is
- **B (ensemble)** — every symbol runs the best-scoring 5-filter ensemble cell from a 3×2×2 grid (K × BOS × exit_type)

Probability gates: 2000-bootstrap, PASS requires P(win)≥0.85, P(PF>1)≥0.75, n≥20, 3/4 quartiles positive, $>0. NEAR relaxes to P(win)≥0.70.


In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))

from dataclasses import replace
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import INSTRUMENTS
from config.deployer import load_all
import research.commod_backtest as cb
from research.ensemble_regime_test import (
    bootstrap, q_pnls, split_stats, grade,
    K_OPTS, BOS_OPTS, EXIT_OPTS, P_WIN_MIN, P_PF1_MIN, N_MIN,
)
from research.current_vs_ensemble import (
    LIVE, EXPANSION, SYM_LEV_EXP, _patch_weekday, _cfg_from_deployed,
    default_cfg, summarize, run_ensemble_grid,
)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 9
print('ready')

In [ ]:
# Load data for all 31 symbols (LIVE + EXPANSION) — uses 4h cache so fast
dep_all = load_all()
all_arrs = {}
lev_map = {}
for sym in LIVE + EXPANSION:
    try:
        d15 = cb.add_features(cb.fetch_hl(sym, '15m', 4000))
        d1h = cb.add_features(cb.fetch_hl(sym, '1h',  2000))
        d4h = cb.add_features(cb.fetch_hl(sym, '4h',  1000))
        if len(d15) < 500: continue
        arr = cb.precompute(d15, d1h, d4h)
        _patch_weekday(arr, sym)
        all_arrs[sym] = arr
        lev_map[sym] = (INSTRUMENTS[sym].hl_max_leverage if sym in LIVE else
                        SYM_LEV_EXP.get(sym, 10)) * 0.15
    except Exception as e:
        print(f'{sym}: {e}')
print(f'loaded {len(all_arrs)} symbols')

In [ ]:
# Run both backtests: CURRENT deployed + best ENSEMBLE cell per symbol.
# Capture trade-level data so we can plot equity curves.
current_results = {}
current_trades = {}
for sym in LIVE:
    dep = dep_all.get(sym)
    if not dep: continue
    cfg = _cfg_from_deployed(dep)
    trades = cb.backtest(all_arrs[sym], cfg, lev_map[sym])
    s = summarize(trades)
    s['entry'] = cfg.entry_type; s['exit'] = cfg.exit_type
    s['filter'] = cfg.trend_filter_1h; s['req4h'] = cfg.require_4h_agreement
    current_results[sym] = s
    current_trades[sym] = trades

ens_results = {}
ens_trades = {}
for sym in all_arrs:
    dep_base = _cfg_from_deployed(dep_all[sym]) if sym in dep_all else default_cfg()
    best, status, _all = run_ensemble_grid(all_arrs[sym], dep_base, lev_map[sym])
    if best is None: continue
    # Re-run the winning cell to capture trades
    if best['exit'] == 'ensemble_hybrid':
        tp1_atr, tp1_pct = 2.0, 0.3
    else:
        tp1_atr, tp1_pct = 0.0, 0.0
    cfg = replace(dep_base,
                  entry_type='ensemble_regime', exit_type=best['exit'],
                  ensemble_k=best['K'], require_bos_confirm=best['bos'],
                  tp1_atr=tp1_atr, tp1_pct=tp1_pct,
                  tp2_atr=0.0, tp3_atr=0.0, trail_atr=0.0, max_hold_bars=1000)
    trades = cb.backtest(all_arrs[sym], cfg, lev_map[sym])
    ens_trades[sym] = trades
    b = dict(best); b['status'] = status
    ens_results[sym] = b

print(f'current: {len(current_results)} syms, ensemble: {len(ens_results)} syms')

## 1. Per-symbol P&L comparison (LIVE universe only)

Side-by-side bars: current deployed vs best ensemble cell per symbol. Green = ensemble wins, red = current wins on OOS $.

In [ ]:
live_rows = []
for sym in LIVE:
    if sym not in current_results or sym not in ens_results: continue
    c = current_results[sym]; e = ens_results[sym]
    live_rows.append({'sym': sym, 'cur_pnl': c['pnl'], 'cur_oos': c['oos_pnl'],
                      'ens_pnl': e['pnl'], 'ens_oos': e['oos_pnl'],
                      'cur_pwin': c['p_win'], 'ens_pwin': e['p_win'],
                      'status': e['status']})
df_live = pd.DataFrame(live_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(df_live)); w = 0.38

ax = axes[0]
b1 = ax.bar(x - w/2, df_live['cur_pnl'], w, label='Current', color='#4C72B0')
b2 = ax.bar(x + w/2, df_live['ens_pnl'], w, label='Ensemble', color='#DD8452')
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(df_live['sym'])
ax.set_ylabel('Total P&L ($)')
ax.set_title('Total P&L — 41d')
ax.legend()

ax = axes[1]
b1 = ax.bar(x - w/2, df_live['cur_oos'], w, label='Current', color='#4C72B0')
b2 = ax.bar(x + w/2, df_live['ens_oos'], w, label='Ensemble', color='#DD8452')
# Highlight winner with green edge on the taller bar
for i, row in df_live.iterrows():
    is_ens_win = row['ens_oos'] > row['cur_oos'] and row['status'] == 'PASS'
    if is_ens_win:
        b2[i].set_edgecolor('green'); b2[i].set_linewidth(2.5)
    else:
        b1[i].set_edgecolor('green'); b1[i].set_linewidth(2.5)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(df_live['sym'])
ax.set_ylabel('OOS P&L ($)')
ax.set_title('OOS P&L — last 30% (green edge = winner)')
ax.legend()

plt.tight_layout(); plt.show()

## 2. Portfolio equity curves

Cumulative P&L over time across all 8 live symbols. Compares three portfolios:
1. **Current** — all 8 live on their deployed configs
2. **Ensemble (strict)** — only the symbols where ensemble PASSES (3 syms)
3. **Hybrid** — recommended: current for 5 symbols, ensemble for 3 winners

In [ ]:
def equity_series(trades_by_sym, include_syms=None):
    rows = []
    for sym, trades in trades_by_sym.items():
        if include_syms is not None and sym not in include_syms: continue
        for t in trades:
            rows.append({'ts': pd.Timestamp(t['ts']), 'pnl': t['pnl']})
    if not rows: return pd.Series(dtype=float)
    df = pd.DataFrame(rows).sort_values('ts')
    df['cum'] = df['pnl'].cumsum()
    return df.set_index('ts')['cum']

cur_eq = equity_series(current_trades)
ens_pass = {s for s, r in ens_results.items() if r['status'] == 'PASS' and s in LIVE}
ens_eq_pass = equity_series(ens_trades, include_syms=ens_pass)

# Hybrid: current for symbols where current wins OOS, ensemble for winners
hybrid_trades = {}
for sym in LIVE:
    if sym not in current_results or sym not in ens_results: continue
    use_ens = (ens_results[sym]['status'] == 'PASS' and
               ens_results[sym]['oos_pnl'] > current_results[sym]['oos_pnl'])
    hybrid_trades[sym] = ens_trades[sym] if use_ens else current_trades[sym]
hyb_eq = equity_series(hybrid_trades)

fig, ax = plt.subplots(figsize=(12, 5))
cur_eq.plot(ax=ax, label=f'Current (8 syms) — end ${cur_eq.iloc[-1]:+.0f}', color='#4C72B0', lw=1.8)
ens_eq_pass.plot(ax=ax, label=f'Ensemble PASS only ({len(ens_pass)} syms) — end ${ens_eq_pass.iloc[-1]:+.0f}', color='#DD8452', lw=1.8)
hyb_eq.plot(ax=ax, label=f'Hybrid (winner-per-symbol) — end ${hyb_eq.iloc[-1]:+.0f}', color='#55A868', lw=2.2)
ax.axhline(0, color='k', lw=0.6)
ax.set_title('Cumulative P&L over 41d'); ax.set_ylabel('P&L ($)')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 3. Drawdown comparison

Peak-to-trough drawdown over time for each portfolio. Smaller = smoother ride.

In [ ]:
def drawdown_pct(eq):
    # Treat starting equity as 10000 for DD % (matches backtester)
    equity = 10000 + eq
    peak = equity.cummax()
    return (equity - peak) / peak * 100

fig, ax = plt.subplots(figsize=(12, 4))
drawdown_pct(cur_eq).plot(ax=ax, label='Current', color='#4C72B0', lw=1.6)
drawdown_pct(ens_eq_pass).plot(ax=ax, label='Ensemble PASS', color='#DD8452', lw=1.6)
drawdown_pct(hyb_eq).plot(ax=ax, label='Hybrid', color='#55A868', lw=2.0)
ax.fill_between(drawdown_pct(hyb_eq).index, drawdown_pct(hyb_eq).values, 0,
                color='#55A868', alpha=0.12)
ax.axhline(0, color='k', lw=0.6)
ax.set_title('Rolling drawdown (%)'); ax.set_ylabel('DD %')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Probability vs OOS P&L scatter

Every tested cell across all 31 symbols. X = bootstrap P(win > 0) on 2000 resamples, Y = OOS $. Red dashed line = the 0.85 probability gate. The upper-right quadrant is the "ship zone".

In [ ]:
# Pull ALL cells (not just best) for every symbol by re-running the grid.
# We just need (p_win, oos_pnl, sym, status) points.
scatter_rows = []
for sym in all_arrs:
    arr = all_arrs[sym]
    dep_base = _cfg_from_deployed(dep_all[sym]) if sym in dep_all else default_cfg()
    lev = lev_map[sym]
    for K, bos, ex in product(K_OPTS, BOS_OPTS, EXIT_OPTS):
        if ex == 'ensemble_hybrid':
            tp1_atr, tp1_pct = 2.0, 0.3
        else:
            tp1_atr, tp1_pct = 0.0, 0.0
        cfg = replace(dep_base,
                      entry_type='ensemble_regime', exit_type=ex,
                      ensemble_k=K, require_bos_confirm=bos,
                      tp1_atr=tp1_atr, tp1_pct=tp1_pct,
                      tp2_atr=0.0, tp3_atr=0.0, trail_atr=0.0, max_hold_bars=1000)
        trades = cb.backtest(arr, cfg, lev)
        s = summarize(trades)
        s['sym'] = sym; s['K'] = K; s['bos'] = bos; s['exit'] = ex
        s['grade'] = grade(s); s['live'] = sym in LIVE
        scatter_rows.append(s)
df_sc = pd.DataFrame(scatter_rows)
df_sc_big = df_sc[df_sc['n'] >= 20].copy()
print(f'cells with n≥20: {len(df_sc_big)} / {len(df_sc)}')

fig, ax = plt.subplots(figsize=(12, 7))
pass_mask = df_sc_big['grade'] == 'PASS'
live_mask = df_sc_big['live']
ax.scatter(df_sc_big.loc[~pass_mask & ~live_mask, 'p_win'],
           df_sc_big.loc[~pass_mask & ~live_mask, 'oos_pnl'],
           s=20, c='lightgray', alpha=0.6, label='Expansion / no pass')
ax.scatter(df_sc_big.loc[~pass_mask & live_mask, 'p_win'],
           df_sc_big.loc[~pass_mask & live_mask, 'oos_pnl'],
           s=40, c='#4C72B0', alpha=0.7, label='Live / no pass')
ax.scatter(df_sc_big.loc[pass_mask, 'p_win'],
           df_sc_big.loc[pass_mask, 'oos_pnl'],
           s=85, c='#55A868', edgecolor='black', label='PASS')

# Label the PASS cells with their symbol
for _, row in df_sc_big[pass_mask].iterrows():
    ax.annotate(f"{row['sym']} K={row['K']}{'/B' if row['bos'] else ''}",
                (row['p_win'], row['oos_pnl']),
                xytext=(8, 6), textcoords='offset points', fontsize=8)

ax.axvline(0.85, color='red', linestyle='--', lw=1, label='P(win)=0.85 gate')
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Bootstrap P(win) — probability total $ > 0')
ax.set_ylabel('OOS P&L ($)')
ax.set_title('Probability–OOS frontier (every cell, n≥20)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5. Ship recommendation table

In [ ]:
rec = []
for sym in LIVE:
    c = current_results[sym]; e = ens_results.get(sym, {})
    winner = 'ENSEMBLE' if (e.get('status') == 'PASS' and e['oos_pnl'] > c['oos_pnl']) else 'CURRENT'
    action = ('SWAP to ensemble' if winner == 'ENSEMBLE' else 'keep')
    cell = (f"K={e['K']}/bos={'T' if e['bos'] else 'F'}/{e['exit'].replace('ensemble_','')}"
            if winner == 'ENSEMBLE' else f"{c['entry']}/{c['filter']}")
    rec.append({'symbol': sym, 'action': action, 'config': cell,
                'cur_oos': c['oos_pnl'], 'ens_oos': e.get('oos_pnl', 0),
                'delta_oos': e.get('oos_pnl', c['oos_pnl']) - c['oos_pnl'] if winner == 'ENSEMBLE' else 0,
                'p_win': e.get('p_win', c['p_win'])})
df_rec = pd.DataFrame(rec)
total_delta = df_rec['delta_oos'].sum()
print(df_rec.to_string(index=False))
print(f'\nExpected OOS uplift from hybrid ship: ${total_delta:+.0f}')